# PhysX-Omni 论文精读：第 3 步 - 核心创新点与实现机制

本 notebook 是第三章的可执行讲义。完整主文档和三个专题 markdown 已经生成在同一目录。本 notebook 重点保留核心结论和一个 toy template-based RLE 演示，帮助把论文表示法转成直观代码。


# PhysX-Omni 论文精读：第 3 步 - 核心创新点与实现机制

这一章聚焦“创新点是什么、为什么重要、分别怎么做”。我把 PhysX-Omni 的创新拆成四个核心层级和两个辅助验证层级：

| 层级 | 创新点 | 创新强度 | 一句话说明 |
|


## 核心创新速览

1. 统一 sim-ready physical 3D generation：从单图生成几何、部件、尺度、材料、affordance、运动学和仿真格式。
2. Template-based RLE：把 part-level 64³ voxel 变成 VLM 可生成的普通文本。
3. PhysXVerse：>8.7K assets、>2.9K categories、part count 1-65 的物理 3D 数据集。
4. PhysX-Bench：用 VLM/渲染图像视频评估 geometry、scale、material、affordance、kinematics、description。
5. Qwen2.5-VL + TRELLIS + URDF/XML：形成可复现工程闭环。


In [1]:
# Toy demo: template-based RLE 如何表达相似的 z-slices。
# 这里用 8x8x4 小网格演示，论文/代码实际用 64x64x64。
import numpy as np

def mask_to_runs(mask):
    flat = mask.astype(np.uint8).ravel()
    runs = []
    i = 0
    while i < len(flat):
        if flat[i] == 0:
            i += 1
            continue
        start = i
        while i < len(flat) and flat[i] == 1:
            i += 1
        runs.append((start, i - start))
    return set(runs)

def fmt(runs):
    items = []
    for s, L in sorted(runs):
        items.append(str(s) if L == 1 else f"{s} {L}")
    return ";".join(items)

# 构造 4 个相似切片：一个小矩形在不同 z 上轻微变化。
slices = []
for z in range(4):
    m = np.zeros((8,8), dtype=np.uint8)
    m[2:5, 2:6] = 1
    if z == 1:
        m[5, 3:5] = 1       # 增加一小块
    if z == 2:
        m[2, 2] = 0         # 删除一个点
    if z == 3:
        m[1:6, 1:7] = 1     # 变化较大，应该新建模板
    slices.append(m)

runs_by_z = [mask_to_runs(m) for m in slices]
templates = []
labels = []
lines = []
threshold = 0.75

def sim(a,b):
    if not a and not b: return 1.0
    if not a or not b: return 0.0
    return len(a & b) / max(len(a), len(b))

for z, runs in enumerate(runs_by_z):
    best = None
    best_sim = -1
    for i, t in enumerate(templates):
        s = sim(runs, t)
        if s > best_sim:
            best = i
            best_sim = s
    if best is None or best_sim < threshold:
        label = chr(ord('a') + len(templates))
        templates.append(runs)
        labels.append(label)
        lines.append(f"{z}: layer {label}")
    else:
        label = labels[best]
        base = templates[best]
        adds = runs - base
        rems = base - runs
        add_txt = f" +[{fmt(adds)}]" if adds else ""
        rem_txt = f" -[{fmt(rems)}]" if rems else ""
        lines.append(f"{z}: layer {label}{add_txt}{rem_txt}")

print("Template definitions:")
for label, t in zip(labels, templates):
    print(f"{label}: {fmt(t)}")
print("\nLayer references:")
for line in lines:
    print(line)


Template definitions:
a: 18 4;26 4;34 4
b: 19 3;26 4;34 4
c: 9 6;17 6;25 6;33 6;41 6

Layer references:
0: layer a
1: layer a +[43 2]
2: layer b
3: layer c


## 产物文件

- `physx_omni_paper_core_innovations_step3.md`：主讲义。
- `physx_omni_step3_innovation_1_template_rle_deepdive.md`：Template-based RLE 深挖。
- `physx_omni_step3_innovation_2_physxverse_physxbench_deepdive.md`：数据集与 benchmark 深挖。
- `physx_omni_step3_innovation_3_pipeline_code_repro_mapping.md`：代码和复现产物对齐。
